# Logistic Regressions

Derive the odds of one stopping and lingering at a third place given their mode of transport, interaction with strangers, recognised regular, same place return frequency, seeing familiar faces, detour tolerance minutes, main stop barriers. 

In [45]:
import numpy as np
import pandas as pd

import statsmodels.api as sm
import statsmodels.formula.api as smf

survey = pd.read_csv("/Users/tonyvo/Desktop/Thesis/survey/survey_recoded.csv")
df = survey.copy()

In [46]:
def make_distribution_table(data, col, label_map=None):
    valid = data[col].dropna()

    counts = valid.value_counts()
    probs = valid.value_counts(normalize=True)

    table = pd.DataFrame({
        "value": counts.index,
        "count": counts.values,
        "probability": probs.values,
        "percentage": probs.values * 100
    })

    if label_map is not None:
        table["label"] = table["value"].map(label_map)
    else:
        table["label"] = table["value"].astype(str)

    return table[["value", "label", "count", "probability", "percentage"]]

In [47]:
needed_cols = [
    # dependent variable
    "third_place_visit_freq",
    "linger_freq",

    # Predictors
    "main_mode",
    "small_interactions_freq",
    "recognized_regular",
    "same_place_return_freq",
    "detour_tolerance_mins",
    "main_stop_barrier",
    "familiar_faces_freq"
]

# Labels
difficulty_labels = {
    1: "Very difficult",
    2: "Difficult",
    3: "Neutral",
    4: "Easy",
    5: "Very easy"
}

visit_freq_labels = {
    1: "Never",
    2: "Less than once a month",
    3: "1-3 times a month",
    4: "1-2 times a week",
    5: "3 or more times a week"
}

frequency_labels = {
    1: "Never",
    2: "Rarely",
    3: "Sometimes",
    4: "Often",
    5: "Very often"
}

detour_tolerance_labels = {
    1: "0 minutes — I would not go out of my way",
    2: "1-5 minutes",
    3: "5-10 minutes",
    4: "10-15 minutes",
    5: "More than 15 minutes",
}


ordinal_label_maps = {
     # dependent variable
    "third_place_visit_freq": visit_freq_labels,
    "linger_freq": frequency_labels,
    "familiar_faces_freq": frequency_labels,

    # Predictors
    "main_mode": None, 
    "small_interactions_freq": frequency_labels,
    "recognized_regular": frequency_labels,
    "same_place_return_freq": frequency_labels,
    "detour_tolerance_mins": detour_tolerance_labels,
    "main_stop_barrier": None
}


for col in needed_cols:
    table = make_distribution_table(
        df,
        col,
        label_map= ordinal_label_maps[col]
    )

    print(f"\n--- Variable tag: {col} ---")
    display(table)


--- Variable tag: third_place_visit_freq ---


,value,label,count,probability,percentage
0,3,1-3 times a month,15,0.454545,45.454545
1,4,1-2 times a week,13,0.393939,39.393939
2,2,Less than once a month,2,0.060606,6.060606
3,1,Never,2,0.060606,6.060606
4,5,3 or more times a week,1,0.030303,3.030303



--- Variable tag: linger_freq ---


,value,label,count,probability,percentage
0,3,Sometimes,15,0.454545,45.454545
1,4,Often,8,0.242424,24.242424
2,2,Rarely,7,0.212121,21.212121
3,5,Very often,2,0.060606,6.060606
4,1,Never,1,0.030303,3.030303



--- Variable tag: main_mode ---


...

,value,label,count,probability,percentage
0,3.0,5-10 minutes,11,0.478261,47.826087
1,4.0,10-15 minutes,8,0.347826,34.782609
2,2.0,1-5 minutes,3,0.130435,13.043478
3,5.0,More than 15 minutes,1,0.043478,4.347826



--- Variable tag: main_stop_barrier ---


,value,label,count,probability,percentage
0,Places are too expensive,Places are too expensive,15,0.454545,45.454545
1,I prefer going straight home,I prefer going straight home,8,0.242424,24.242424
2,Lack of time,Lack of time,7,0.212121,21.212121
3,Places are too far away,Places are too far away,3,0.090909,9.090909



--- Variable tag: familiar_faces_freq ---


,value,label,count,probability,percentage
0,3,Sometimes,13,0.393939,39.393939
1,2,Rarely,10,0.303030,30.303030
2,4,Often,6,0.181818,18.181818
3,5,Very often,2,0.060606,6.060606
4,1,Never,2,0.060606,6.060606


In [48]:
# Make dependent variables binary
df["stop_binary"] = df["third_place_visit_freq"].map({
    # If your column contains numeric encoded values
    1: 0,
    2: 0,
    3: 0,
    4: 1,
    5: 1
})

df["linger_binary"] = df["linger_freq"].map({
    1: 0,
    2: 0,
    3: 0,
    4: 1,
    5: 1
})

print(df["linger_binary"].describe())

count    33.000000
mean      0.303030
std       0.466694
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000
Name: linger_binary, dtype: float64


In [ ]:
def run_logistic_model(
    df,
    dependent_var,
    model_name,
    *predictors,
    categorical_predictors=None
):
    """
    Runs a logistic regression with an arbitrary number of predictors.

    Parameters
    ----------
    df : pandas DataFrame
    dependent_var : str
        Binary dependent variable, e.g. "stop_binary" or "linger_binary"
    model_name : str
        Name of the model
    *predictors : str
        Any number of predictor column names
    categorical_predictors : list, optional
        Predictors that should be dummy-coded, e.g. ["main_mode", "main_stop_barrier"]
    """

    if categorical_predictors is None:
        categorical_predictors = []

    predictors = list(predictors)

    model_vars = [dependent_var] + predictors
    model_df = df[model_vars].dropna().copy()

    print("\n==============================")
    print(model_name)
    print("==============================")
    print("Rows used:", len(model_df))
    print("Dependent variable distribution:")
    print(model_df[dependent_var].value_counts(dropna=False))

    if len(model_df) == 0:
        raise ValueError("No rows left after dropna(). Check your variables.")

    X = model_df[predictors].copy()
    y = model_df[dependent_var].astype(float)

    # Dummy-code categorical predictors
    if len(categorical_predictors) > 0:
        X = pd.get_dummies(
            X,
            columns=categorical_predictors,
            drop_first=True,
            dtype=float
        )

    # Convert all remaining columns to numeric
    X = X.astype(float)

    # Drop predictors with no variation
    zero_variance_cols = X.columns[X.nunique() <= 1].tolist()
    if zero_variance_cols:
        print("Dropping zero-variance predictors:", zero_variance_cols)
        X = X.drop(columns=zero_variance_cols)

    X = sm.add_constant(X)

    model = sm.Logit(y, X)
    result = model.fit(method="lbfgs", maxiter=1000)

    odds_table = pd.DataFrame({
        "model": model_name,
        "variable": result.params.index,
        "coef_log_odds": result.params.values, # returns raw Beta coeefficient
        "odds_ratio": np.exp(result.params.values), # returns e^Beta which is the odds ratio
        "p_value": result.pvalues.values,
        "ci_lower": np.exp(result.conf_int()[0].values),
        "ci_upper": np.exp(result.conf_int()[1].values)
    })

    odds_table = odds_table.sort_values("odds_ratio", ascending=False)

    return result, odds_table

In [ ]:
stop_result, stop_odds = run_logistic_model(
    df,
    "stop_binary", # dependent variable
    "Model 1: Stopping vs Recognised Regular", # Model Name

    # Predictors
    "recognized_regular",
    categorical_predictors=None
)
display(stop_odds)


Model 1: Stopping
Rows used: 33
Dependent variable distribution:
stop_binary
0    19
1    14
Name: count, dtype: int64


,model,variable,coef_log_odds,odds_ratio,p_value,ci_lower,ci_upper
1,Model 1: Stopping,recognized_regular,0.589014,1.802210,0.042474,1.020199,3.183653
0,Model 1: Stopping,const,-2.238146,0.106656,0.033799,0.013502,0.842522


In [80]:
stop_result, stop_odds = run_logistic_model(
    df,
    "stop_binary", # dependent variable
    "Model 2: Stopping vs Frequency of Seeing Familiar Faces", # Model Name

    # Predictors
    "familiar_faces_freq",
    categorical_predictors=None
)
display(stop_odds)


Model 2: Stopping vs Frequency of Seeing Familiar Faces
Rows used: 33
Dependent variable distribution:
stop_binary
0    19
1    14
Name: count, dtype: int64


,model,variable,coef_log_odds,odds_ratio,p_value,ci_lower,ci_upper
1,Model 2: Stopping vs Frequency of Seeing Famil...,familiar_faces_freq,1.027854,2.795061,0.027142,1.123079,6.956205
0,Model 2: Stopping vs Frequency of Seeing Famil...,const,-3.315593,0.036313,0.020325,0.002207,0.597569


In [86]:
stop_result, stop_odds = run_logistic_model(
    df,
    "stop_binary", # dependent variable
    "Model 3: Stopping vs Too Rushed", # Model Name

    # Predictors
    "too_rushed",
    categorical_predictors=None
)
display(stop_odds)


Model 3: Stopping vs Too Rushed
Rows used: 33
Dependent variable distribution:
stop_binary
0    19
1    14
Name: count, dtype: int64


,model,variable,coef_log_odds,odds_ratio,p_value,ci_lower,ci_upper
0,Model 3: Stopping vs Too Rushed,const,2.550344,12.811505,0.055140,0.945431,173.608304
1,Model 3: Stopping vs Too Rushed,too_rushed,-0.919325,0.398788,0.028567,0.175116,0.908153


In [130]:
stop_result, stop_odds = run_logistic_model(
    df,
    "linger_binary", # dependent variable
    "Model 4: Linger vs ", # Model Name

    # Predictors
    "schedule_fixedness",
    categorical_predictors=None
)
display(stop_odds)


Model 4: Linger vs 
Rows used: 33
Dependent variable distribution:
linger_binary
0    23
1    10
Name: count, dtype: int64


,model,variable,coef_log_odds,odds_ratio,p_value,ci_lower,ci_upper
1,Model 4: Linger vs,schedule_fixedness,0.141991,1.152566,0.728666,0.516723,2.570829
0,Model 4: Linger vs,const,-1.335408,0.263051,0.375459,0.013725,5.041645
